# Video Studio — `wan21-1-3b` on Colab (manual backup)

Kaggle is the studio's automated GPU provider. Use this notebook only when Kaggle has no GPU free.

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Stay on the page.
2. When it finishes, the browser downloads **`vs-result-wan21-1-3b.zip`**. Give that file to Claude
   (upload it to the session or commit it to the branch); Claude runs `vs-colab import`.
3. The last cell releases the GPU so your Colab allowance is not wasted.

Colab rules: run it yourself in this notebook UI; no SSH, remote desktop or web UIs; one account;
no deepfakes — lip sync only on faces you own, AI-generated people, or people who agreed in writing.
Free Colab limits are not published and change; a run can be stopped if you are idle.

Same code as the Kaggle job (packed by `vs-colab build` from `gpu-jobs/`); models are downloaded
from their official Hugging Face / GitHub sources at pinned revisions.

In [ ]:
# 1. GPU check + map the Kaggle paths the job uses onto /content
import os, subprocess, pathlib
r = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert r.returncode == 0, 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again'
print(r.stdout)
for d in ('/content/kaggle/working', '/content/kaggle/temp'):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
if not os.path.exists('/kaggle'):
    os.symlink('/content/kaggle', '/kaggle')


In [ ]:
# 3. the studio job (generated — do not edit; rebuild with vs-colab build)
JOB = '# --- packed by vs-gpu: inputs, params, shared runtime (gpu-jobs/_common.py), then the job ---\nimport base64, json, os, pathlib, time\n_T0 = time.time()\nIN = pathlib.Path("/kaggle/working/in"); OUT = pathlib.Path("/kaggle/working/out")\nIN.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)\nPARAMS = json.loads(\'{"model": "wan21-1.3b", "preset": "t4", "cfg_until": 0.5, "item_budget_s": 900, "max_job_s": 2400, "items": [{"id": "kite", "prompt": "a red kite flying over green hills on a sunny day, clouds drifting, slow pan right, cinematic", "seed": 3}, {"id": "kite-fbc", "prompt": "a red kite flying over green hills on a sunny day, clouds drifting, slow pan right, cinematic", "cache": 0.05, "seed": 3}], "_need_gpu": true, "_need_internet": true, "parallel": false}\')\nfor _n, _b in json.loads(\'{}\').items():\n    (IN / _n).write_bytes(base64.b64decode(_b))\n_COMMON_B64 = \'IyBTaGFyZWQgcnVudGltZSBmb3IgZXZlcnkgc3R1ZGlvIEdQVSBqb2IgKHBhY2tlZCBieSB2cy1ncHU7IHJ1bnMgaW5zaWRlIHRoZSBLYWdnbGUga2VybmVsIGFuZCBpdHMKIyBwZXItR1BVIGNoaWxkIHByb2Nlc3NlcykuIFByb3ZpZGVzOiByZXBvcnQoKSwgZ3B1X2luZm8oKSwgaXNfdDQoKSwgcGlja19kdHlwZSgpLCBzaGFyZCgpLCBsb2FkX3BpcGUoKS4KIyBLYWdnbGUgZnJlZSB0aWVyID0gMnggVGVzbGEgVDQgKHNtNzUsIDE0LjYgR0IgZWFjaCk6IGZhc3QgZnAxNiB0ZW5zb3IgY29yZXMsIGJmMTYgb25seSBlbXVsYXRlZC4KaW1wb3J0IGpzb24sIG9zLCBwYXRobGliLCBzdWJwcm9jZXNzLCBzeXMsIHRpbWUKClNIQVJEID0gaW50KG9zLmVudmlyb24uZ2V0KCJWU19TSEFSRCIsICItMSIpKSAgICAgICAgICAjID49IDAgaW5zaWRlIGEgcGVyLUdQVSBjaGlsZCBwcm9jZXNzCgoKZGVmIHJlcG9ydCgqKmt3KToKICAgIHAgPSBPVVQgLyAoInJlcG9ydC5qc29uIiBpZiBTSEFSRCA8IDAgZWxzZSBmInJlcG9ydC5zaGFyZHtTSEFSRH0uanNvbiIpCiAgICBkID0ganNvbi5sb2FkcyhwLnJlYWRfdGV4dCgpKSBpZiBwLmV4aXN0cygpIGVsc2Uge30KICAgIGQudXBkYXRlKGt3KTsgZFsiZWxhcHNlZF9zIl0gPSByb3VuZCh0aW1lLnRpbWUoKSAtIF9UMCwgMSk7IHAud3JpdGVfdGV4dChqc29uLmR1bXBzKGQsIGluZGVudD0yKSkKCgpkZWYgX3ZyYW1fcGVhaygpOiAgICAgICAgICAgIyBiZW5jaG1hcms6IHBlYWsgVlJBTSBwZXIgdmlzaWJsZSBHUFUsIHdyaXR0ZW4gYXQgZXhpdCBvZiBldmVyeSAoY2hpbGQpIHByb2Nlc3MKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICByZXBvcnQoKip7InZyYW1fcGVha19nYiI6IFtyb3VuZCh0b3JjaC5jdWRhLm1heF9tZW1vcnlfYWxsb2NhdGVkKGkpIC8gMioqMzAsIDIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGZvciBpIGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpXX0pCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKCgpkZWYgX3dhdGNoZG9nKGxpbWl0X3MpOiAgICAjIGhhcmQgam9iIGJ1ZGdldDogbmV2ZXIgbGV0IGEgc3R1Y2sgam9iIGJ1cm4gR1BVIHF1b3RhIChLYWdnbGUgYWxsb3dzIDEyIGgpCiAgICBpbXBvcnQgdGhyZWFkaW5nCiAgICBkZWYgZmlyZSgpOgogICAgICAgIHJlcG9ydChlcnJvcj1mImpvYiBleGNlZWRlZCBpdHMgYnVkZ2V0IG9mIHtsaW1pdF9zfSBzIGFuZCB3YXMgc3RvcHBlZCAocmFpc2UgbWF4X2pvYl9zIG9ubHkgaWYgbmVlZGVkKSIsCiAgICAgICAgICAgICAgIHRpbWVkX291dD1UcnVlKQogICAgICAgIF92cmFtX3BlYWsoKTsgb3MuX2V4aXQoNSkKICAgIHQgPSB0aHJlYWRpbmcuVGltZXIobGltaXRfcywgZmlyZSk7IHQuZGFlbW9uID0gVHJ1ZTsgdC5zdGFydCgpCgoKaW1wb3J0IGF0ZXhpdAphdGV4aXQucmVnaXN0ZXIoX3ZyYW1fcGVhaykKaWYgU0hBUkQgPCAwIGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOgogICAgX3dhdGNoZG9nKGludChQQVJBTVMuZ2V0KCJtYXhfam9iX3MiLCAzNjAwKSkpCgoKZGVmIHN0ZXBfZ3VhcmQodG90YWxfc3RlcHMsIGJ1ZGdldF9zLCBsYWJlbD0iaXRlbSIsIHdlaWdodHM9Tm9uZSk6CiAgICAiIiJkaWZmdXNlcnMgY2FsbGJhY2tfb25fc3RlcF9lbmQ6IHN0b3Agb24gTmFOL2luZiBsYXRlbnRzIGFuZCB3aGVuIHRoZSBwcm9qZWN0ZWQgdGltZSBvZiB0aGlzIGl0ZW0gZXhjZWVkcwogICAgYnVkZ2V0X3MgKGVhcmx5IGZhaWx1cmUgZGV0ZWN0aW9uLCBtZWFzdXJlZCBwZXIgc3RlcCkuIHdlaWdodHMgPSByZWxhdGl2ZSBjb3N0IHBlciBzdGVwICgyIHdpdGggQ0ZHLCAxCiAgICB3aXRob3V0KSBzbyB0aGUgcHJvamVjdGlvbiBzdGF5cyByaWdodCB3aGVuIENGRyBpcyBzd2l0Y2hlZCBvZmYgcGFydC13YXkuIFJhaXNlcyBGbG9hdGluZ1BvaW50RXJyb3IgLwogICAgVGltZW91dEVycm9yLiIiIgogICAgdDAgPSB0aW1lLnRpbWUoKQogICAgdyA9IGxpc3Qod2VpZ2h0cyBvciBbMV0gKiB0b3RhbF9zdGVwcykKICAgIHN0YXRzID0geyJzdGVwcyI6IDAsICJkZW5vaXNlX3MiOiAwLjB9CiAgICBkZWYgY2IocGlwZSwgc3RlcCwgdHMsIGt3KToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBsYXQgPSBrdy5nZXQoImxhdGVudHMiKQogICAgICAgIGlmIGxhdCBpcyBub3QgTm9uZSBhbmQgbm90IHRvcmNoLmlzZmluaXRlKGxhdCkuYWxsKCk6CiAgICAgICAgICAgIHJhaXNlIEZsb2F0aW5nUG9pbnRFcnJvcihmIntsYWJlbH06IG5vbi1maW5pdGUgbGF0ZW50cyBhdCBzdGVwIHtzdGVwfSIpCiAgICAgICAgZWwgPSB0aW1lLnRpbWUoKSAtIHQwCiAgICAgICAgcHJvaiA9IGVsIC8gc3VtKHdbOnN0ZXAgKyAxXSkgKiBzdW0odykKICAgICAgICBpZiBzdGVwID49IDEgYW5kIHByb2ogPiBidWRnZXRfczoKICAgICAgICAgICAgcmFpc2UgVGltZW91dEVycm9yKGYie2xhYmVsfToge2VsIC8gKHN0ZXAgKyAxKTouMWZ9IHMvc3RlcCwgcHJvamVjdGVkIHtwcm9qOi4wZn0gcyA+IGJ1ZGdldCB7YnVkZ2V0X3N9IHMiKQogICAgICAgIHN0YXRzLnVwZGF0ZShzdGVwcz1zdGVwICsgMSwgZGVub2lzZV9zPXJvdW5kKGVsLCAxKSkKICAgICAgICByZXR1cm4ga3cKICAgIGNiLnN0YXRzID0gc3RhdHMKICAgIHJldHVybiBjYgoKCmRlZiBncHVfaW5mbygpOgogICAgaWYgb3MuZW52aXJvbi5nZXQoIlZTX0ZBS0VfR1BVUyIpOiAgICAgICAjIENQVSBzZWxmLXRlc3Qgb2YgcGFja2luZyBhbmQgZmFuLW91dCAodGVzdHMvdGVzdF9ncHVfcnVudGltZS5zaCkKICAgICAgICByZXR1cm4geyJuIjogaW50KG9zLmVudmlyb25bIlZTX0ZBS0VfR1BVUyJdKSwgIm5hbWUiOiAiZmFrZSIsICJ2cmFtX2diIjogMTQuNiwgImNjIjogNy41fQogICAgaW1wb3J0IHRvcmNoCiAgICBuID0gdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKQogICAgaWYgbm90IG46CiAgICAgICAgcmV0dXJuIHsibiI6IDB9CiAgICBwID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX3Byb3BlcnRpZXMoMCkKICAgIHJldHVybiB7Im4iOiBuLCAibmFtZSI6IHAubmFtZSwgInZyYW1fZ2IiOiByb3VuZChwLnRvdGFsX21lbW9yeSAvIDIqKjMwLCAxKSwgImNjIjogcC5tYWpvciArIHAubWlub3IgLyAxMH0KCgpkZWYgaXNfdDQoKToKICAgIGcgPSBncHVfaW5mbygpCiAgICByZXR1cm4gZy5nZXQoIm4iLCAwKSA+IDAgYW5kIGdbImNjIl0gPCA4LjAKCgpkZWYgcGlja19kdHlwZShmcDE2X3NhZmUpOgogICAgIiIiZnAxNiBvbiBUNCBmb3IgbW9kZWxzIGtub3duIHRvIGJlIGZwMTYtc2FmZSAoNC04eCBmYXN0ZXIgdGhhbiBlbXVsYXRlZCBiZjE2KTsgYmYxNiBvdGhlcndpc2UuCiAgICBQQVJBTVNbImR0eXBlIl0gPSBmcDE2IHwgYmYxNiB8IGZwMzIgb3ZlcnJpZGVzLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICB3YW50ID0gUEFSQU1TLmdldCgiZHR5cGUiLCAiYXV0byIpCiAgICBpZiB3YW50IGluICgiZnAxNiIsICJiZjE2IiwgImZwMzIiKToKICAgICAgICByZXR1cm4geyJmcDE2IjogdG9yY2guZmxvYXQxNiwgImJmMTYiOiB0b3JjaC5iZmxvYXQxNiwgImZwMzIiOiB0b3JjaC5mbG9hdDMyfVt3YW50XQogICAgaWYgaXNfdDQoKToKICAgICAgICByZXR1cm4gdG9yY2guZmxvYXQxNiBpZiBmcDE2X3NhZmUgZWxzZSB0b3JjaC5iZmxvYXQxNgogICAgcmV0dXJuIHRvcmNoLmJmbG9hdDE2CgoKY2xhc3MgX0RvbmUoRXhjZXB0aW9uKToKICAgIHBhc3MKCgpkZWYgc2hhcmQoaXRlbXMpOgogICAgIiIiU21hbGwgbW9kZWxzOiBvbmUgcHJvY2VzcyBwZXIgR1BVLCBpdGVtcyBzcGxpdCByb3VuZC1yb2JpbiAoMnggdGhyb3VnaHB1dCBvbiAyeCBUNCkuCiAgICBSZXR1cm5zIHRoaXMgcHJvY2VzcydzIGl0ZW1zOyBpbiB0aGUgcGFyZW50IGl0IHdhaXRzIGZvciB0aGUgY2hpbGRyZW4sIG1lcmdlcyByZXBvcnRzIGFuZCBzdG9wcyB0aGUgam9iLiIiIgogICAgbiA9IGdwdV9pbmZvKCkuZ2V0KCJuIiwgMCkKICAgIGlmIFNIQVJEID49IDA6CiAgICAgICAgcmV0dXJuIGl0ZW1zW1NIQVJEOjppbnQob3MuZW52aXJvblsiVlNfTlNIQVJEIl0pXQogICAgaWYgbiA8IDIgb3IgbGVuKGl0ZW1zKSA8IDIgb3Igbm90IFBBUkFNUy5nZXQoInBhcmFsbGVsIiwgVHJ1ZSk6CiAgICAgICAgcmV0dXJuIGl0ZW1zCiAgICB0bXAgPSBwYXRobGliLlBhdGgoIi9rYWdnbGUvdGVtcCIpOyB0bXAubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgKHRtcCAvICJ2c19wYXJhbXMuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhQQVJBTVMpKQogICAgKHRtcCAvICJ2c19jaGlsZC5weSIpLndyaXRlX3RleHQoCiAgICAgICAgImltcG9ydCBiYXNlNjQsIGpzb24sIHBhdGhsaWIsIHRpbWVcbl9UMCA9IHRpbWUudGltZSgpXG4iCiAgICAgICAgIklOID0gcGF0aGxpYi5QYXRoKCcva2FnZ2xlL3dvcmtpbmcvaW4nKTsgT1VUID0gcGF0aGxpYi5QYXRoKCcva2FnZ2xlL3dvcmtpbmcvb3V0JylcbiIKICAgICAgICAiUEFSQU1TID0ganNvbi5sb2FkKG9wZW4oJy9rYWdnbGUvdGVtcC92c19wYXJhbXMuanNvbicpKVxuIgogICAgICAgIGYiX0NPTU1PTl9CNjQgPSB7X0NPTU1PTl9CNjQhcn1cbl9KT0JfQjY0ID0ge19KT0JfQjY0IXJ9XG4iCiAgICAgICAgImV4ZWMoYmFzZTY0LmI2NGRlY29kZShfQ09NTU9OX0I2NCkuZGVjb2RlKCkpXG4iCiAgICAgICAgImV4ZWMoY29tcGlsZShiYXNlNjQuYjY0ZGVjb2RlKF9KT0JfQjY0KS5kZWNvZGUoKSwgJ2pvYi5weScsICdleGVjJykpXG4iKQogICAgcHJvY3MgPSBbc3VicHJvY2Vzcy5Qb3Blbihbc3lzLmV4ZWN1dGFibGUsIHN0cih0bXAgLyAidnNfY2hpbGQucHkiKV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGVudj17Kipvcy5lbnZpcm9uLCAiVlNfU0hBUkQiOiBzdHIoaSksICJWU19OU0hBUkQiOiBzdHIobiksICJDVURBX1ZJU0lCTEVfREVWSUNFUyI6IHN0cihpKX0pCiAgICAgICAgICAgICBmb3IgaSBpbiByYW5nZShuKV0KICAgIGNvZGVzID0gW3Aud2FpdCgpIGZvciBwIGluIHByb2NzXQogICAgc2hhcmRzID0gW10KICAgIGZvciBpIGluIHJhbmdlKG4pOgogICAgICAgIGYgPSBPVVQgLyBmInJlcG9ydC5zaGFyZHtpfS5qc29uIgogICAgICAgIGlmIGYuZXhpc3RzKCk6CiAgICAgICAgICAgIHNoYXJkcy5hcHBlbmQoanNvbi5sb2FkcyhmLnJlYWRfdGV4dCgpKSk7IGYudW5saW5rKCkKICAgIHJlcG9ydChwYXJhbGxlbF9ncHVzPW4sIHNoYXJkcz1zaGFyZHMsIHNoYXJkX2V4aXRfY29kZXM9Y29kZXMpCiAgICBpZiBjb2RlcyBhbmQgYWxsKGNvZGVzKToKICAgICAgICByZXBvcnQoZXJyb3I9ZiJldmVyeSBHUFUgd29ya2VyIGZhaWxlZDogZXhpdCBjb2RlcyB7Y29kZXN9IikKICAgIGVsaWYgYW55KGNvZGVzKTogICAgICAgICAgICAgICAgICAgICAgICAgICMgcGFydGlhbCByZXN1bHQ6IGtlZXAgd2hhdCB3YXMgbWFkZSwgdnMtcHJvZHVjZSBmYWxscyBiYWNrIGZvciB0aGUgcmVzdAogICAgICAgIHJlcG9ydCh3YXJuaW5nPWYiYSBHUFUgd29ya2VyIGZhaWxlZDogZXhpdCBjb2RlcyB7Y29kZXN9IikKICAgIHJhaXNlIF9Eb25lKCkKCgpkZWYgbG9hZF9waXBlKGNscywgcmVwbywgcmV2aXNpb24sIGR0eXBlLCBmcDhfb2s9VHJ1ZSwgcHJlZmVyPU5vbmUsICoqa3cpOgogICAgIiIiQmlnIGRpZmZ1c2lvbiBtb2RlbHMgb24gMnggVDQsIGJlc3Qgc3RyYXRlZ3kgZmlyc3Q6CiAgICAgICAxKSBkZXZpY2VfbWFwPSdiYWxhbmNlZCcg4oCUIHRleHQgZW5jb2RlciBvbiBvbmUgVDQsIHRyYW5zZm9ybWVyICsgVkFFIG9uIHRoZSBvdGhlciAobm8gQ1BVIHJvdW5kLXRyaXBzKQogICAgICAgMikgZnA4IHdlaWdodCBzdG9yYWdlIGZvciB0aGUgdHJhbnNmb3JtZXIgKyBtb2RlbCBDUFUgb2ZmbG9hZCBvbiBvbmUgVDQKICAgICAgIDMpIHNlcXVlbnRpYWwgQ1BVIG9mZmxvYWQgKHNsb3dlc3QsIGFsd2F5cyBmaXRzKQogICAgUEFSQU1TWyJwbGFjZW1lbnQiXSA9IGJhbGFuY2VkIHwgb2ZmbG9hZCB8IHNlcXVlbnRpYWwgZm9yY2VzIG9uZS4iIiIKICAgIGltcG9ydCB0b3JjaAogICAgZm9yY2UgPSBwcmVmZXIgb3IgUEFSQU1TLmdldCgicGxhY2VtZW50IikKICAgIGVycm9ycyA9IFtdCiAgICBpZiAoZm9yY2UgaW4gKE5vbmUsICJiYWxhbmNlZCIpKSBhbmQgZ3B1X2luZm8oKS5nZXQoIm4iLCAwKSA+PSAyOgogICAgICAgIHRyeToKICAgICAgICAgICAgcGlwZSA9IGNscy5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2aXNpb24sIHRvcmNoX2R0eXBlPWR0eXBlLCBkZXZpY2VfbWFwPSJiYWxhbmNlZCIsICoqa3cpCiAgICAgICAgICAgIGZwOCA9IGZwOF9vayBhbmQgZ3B1X2luZm8oKVsidnJhbV9nYiJdIDwgMjAgYW5kIGhhc2F0dHIocGlwZSwgInRyYW5zZm9ybWVyIikKICAgICAgICAgICAgaWYgZnA4OiAgICMgbWVhc3VyZWQ6IFotSW1hZ2UncyA2QiB0cmFuc2Zvcm1lciBhbG9uZSAoMTIgR0IgYmYxNikgKyBhdHRlbnRpb24gb3ZlcmZsb3dzIGEgMTQuNiBHQiBUNAogICAgICAgICAgICAgICAgcGlwZS50cmFuc2Zvcm1lci5lbmFibGVfbGF5ZXJ3aXNlX2Nhc3Rpbmcoc3RvcmFnZV9kdHlwZT10b3JjaC5mbG9hdDhfZTRtM2ZuLCBjb21wdXRlX2R0eXBlPWR0eXBlKQogICAgICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgICAgIHJlcG9ydChwbGFjZW1lbnQ9ImJhbGFuY2VkICgyIEdQVXMpIiArICgiICsgZnA4IHRyYW5zZm9ybWVyIHdlaWdodHMiIGlmIGZwOCBlbHNlICIiKSkKICAgICAgICAgICAgcmV0dXJuIHBpcGUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJiYWxhbmNlZDoge3R5cGUoZSkuX19uYW1lX199OiB7c3RyKGUpWzoyMDBdfSIpCiAgICBpZiBmb3JjZSBpbiAoTm9uZSwgIm9mZmxvYWQiKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHBpcGUgPSBjbHMuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldmlzaW9uLCB0b3JjaF9kdHlwZT1kdHlwZSwgKiprdykKICAgICAgICAgICAgaWYgZnA4X29rIGFuZCBncHVfaW5mbygpWyJ2cmFtX2diIl0gPCAyMCBhbmQgaGFzYXR0cihwaXBlLCAidHJhbnNmb3JtZXIiKToKICAgICAgICAgICAgICAgIHBpcGUudHJhbnNmb3JtZXIuZW5hYmxlX2xheWVyd2lzZV9jYXN0aW5nKHN0b3JhZ2VfZHR5cGU9dG9yY2guZmxvYXQ4X2U0bTNmbiwgY29tcHV0ZV9kdHlwZT1kdHlwZSkKICAgICAgICAgICAgcGlwZS5lbmFibGVfbW9kZWxfY3B1X29mZmxvYWQoZ3B1X2lkPTApCiAgICAgICAgICAgIHJlcG9ydChwbGFjZW1lbnQ9ImZwOCB0cmFuc2Zvcm1lciArIG1vZGVsIENQVSBvZmZsb2FkICgxIEdQVSkiLCBwbGFjZW1lbnRfZXJyb3JzPWVycm9ycykKICAgICAgICAgICAgcmV0dXJuIHBpcGUKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgICAgIGVycm9ycy5hcHBlbmQoZiJvZmZsb2FkOiB7dHlwZShlKS5fX25hbWVfX306IHtzdHIoZSlbOjIwMF19IikKICAgIHBpcGUgPSBjbHMuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldmlzaW9uLCB0b3JjaF9kdHlwZT1kdHlwZSwgKiprdykKICAgIHBpcGUuZW5hYmxlX3NlcXVlbnRpYWxfY3B1X29mZmxvYWQoKQogICAgcmVwb3J0KHBsYWNlbWVudD0ic2VxdWVudGlhbCBDUFUgb2ZmbG9hZCIsIHBsYWNlbWVudF9lcnJvcnM9ZXJyb3JzKQogICAgcmV0dXJuIHBpcGUKCgpkZWYgdmFlX3RpbGluZyhwaXBlKToKICAgIGZvciBuYW1lIGluICgidmFlIiwpOgogICAgICAgIG0gPSBnZXRhdHRyKHBpcGUsIG5hbWUsIE5vbmUpCiAgICAgICAgZm9yIGZuIGluICgiZW5hYmxlX3RpbGluZyIsICJlbmFibGVfc2xpY2luZyIpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBnZXRhdHRyKG0sIGZuKSgpCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCgoKZGVmIGxvb2tzX2Jyb2tlbihhcnIpOgogICAgIiIiTmFOL2JsYWNrIG91dHB1dCBjaGVjayAoZnAxNiBvdmVyZmxvdyBzaG93cyB1cCBhcyBibGFjayBmcmFtZXMpLiIiIgogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBhID0gbnAuYXNhcnJheShhcnIsIGR0eXBlPW5wLmZsb2F0MzIpCiAgICBpZiBub3QgbnAuaXNmaW5pdGUoYSkuYWxsKCk6CiAgICAgICAgcmV0dXJuIFRydWUKICAgIGlmIGEuc2l6ZSBhbmQgZmxvYXQoYS5tYXgoKSkgPD0gMS4wOiAgICAgICMgcGlwZWxpbmVzIHJldHVybmluZyBmbG9hdHMgaW4gWzAsIDFdIChlLmcuIFdhbiB2aWRlbyBmcmFtZXMpCiAgICAgICAgYSA9IGEgKiAyNTUuMAogICAgcmV0dXJuIGZsb2F0KGEubWVhbigpKSA8IDIuMAo=\'\n_JOB_B64 = \'IyBHUFUgam9iOiB0ZXh0LXRvLXZpZGVvIGFuZCBpbWFnZS10by12aWRlbywgdHVuZWQgZm9yIEthZ2dsZSdzIDJ4IFRlc2xhIFQ0ICgxNC42IEdCIGVhY2gsIHNtNzUpLgojIFBBUkFNUzogeyJtb2RlbCI6ICJ3YW4yMS0xLjNiIiB8ICJ3YW4yMi01YiIsICJwcmVzZXQiOiAidDQiIHwgInQ0LWZhc3QiIHwgImhxIiwgImFsbG93X2hlYXZ5IjogZmFsc2UsCiMgICAgICAgICAgIml0ZW1fYnVkZ2V0X3MiOiAxNTAwLCAibWF4X2pvYl9zIjogNTQwMCwKIyAgICAgICAgICAiaXRlbXMiOiBbeyJpZCIsInByb21wdCIsImltYWdlIj8gKGtleWZyYW1lIGluIElOLyAtPiBpbWFnZS10by12aWRlbyksImZyYW1lcyIsIndpZHRoIiwiaGVpZ2h0Iiwic2VlZCJ9XSwKIyAgICAgICAgICAic3RlcHMiOiBOLCAiZnBzIjogMjQsICJkdHlwZSI6ICJhdXRvInwiZnAxNiJ8ImJmMTYifQojCiMgICB3YW4yMS0xLjNiIERFRkFVTFQgb24gVDQuIFdhbjIuMS1UMlYtMS4zQiAoQXBhY2hlLTIuMCk6IGxpZ2h0IHRleHQtdG8tdmlkZW8gYXQgNDgwcDsgdGV4dC1vbmx5LgojICAgd2FuMjItNWIgICBBRFZBTkNFRCAob3B0aW9uYWwsIGFsbG93X2hlYXZ5KS4gV2FuMi4yLVRJMlYtNUIgKEFwYWNoZS0yLjApOiB0ZXh0LSBBTkQgaW1hZ2UtdG8tdmlkZW8uCiMgICAgICAgICAgICAgIE1lYXN1cmVkIG9uIFQ0IChiYWxhbmNlZCArIGZwOCk6IDcuOCBzL3N0ZXAsIFZBRSBkZWNvZGUgb2ZmLUdQVSB+NjIgbWluLCBibGFjayBmcDE2IG91dHB1dC4KIyAgIEJvdGg6IG9uZSBUNCwgbW9kZWwgQ1BVIG9mZmxvYWQgKFVNVDUgZW5jb2RlcyB0aGVuIHN3YXBzIG91dCksIGZwMTYgY29tcHV0ZSAoVDQgaGFzIG5vIG5hdGl2ZSBiZjE2KSwKIyAgIFZBRSB0aWxpbmcsIHBlci1zdGVwIGd1YXJkIChOYU4gbGF0ZW50cyBvciBwcm9qZWN0ZWQgdGltZSA+IGl0ZW1fYnVkZ2V0X3Mgc3RvcHMgdGhlIGNsaXAgZWFybHkpLAojICAgY2xpcHMgd2l0aCBOYU4gYXJlIHJlZG9uZSBvbmNlIGluIGJmMTY7IGhhcmQgam9iIGJ1ZGdldCBtYXhfam9iX3MuCiMgICBMb3cgaG9zdCBSQU0gKDwgMjAgR0IsIGUuZy4gZnJlZSBDb2xhYidzIDEyLjcgR0I7IG9yIFBBUkFNUyBsb3dfcmFtKTogdGhlIDExIEdCIFVNVDUgdGV4dCBlbmNvZGVyIHdvdWxkIG5vdAojICAgZml0IGluIFJBTSBuZXh0IHRvIHRoZSB0cmFuc2Zvcm1lciBmb3IgQ1BVIG9mZmxvYWQsIHNvIGFsbCBwcm9tcHRzIGFyZSBlbmNvZGVkIGZpcnN0IHdpdGggdGhlIGVuY29kZXIgb24gdGhlCiMgICBHUFUsIHRoZSBlbmNvZGVyIGlzIGZyZWVkLCBhbmQgdGhlIHBpcGVsaW5lIGlzIGJ1aWx0IHdpdGhvdXQgaXQgKHByb21wdCBlbWJlZGRpbmdzIHBhc3NlZCBpbikuCiMKIyBXYW4yLjEtMS4zQiBvbiBUNCAobWVhc3VyZWQgMjAyNi0xMS0yNSwgODMyeDQ4MHg4MSwgZnVsbCBDRkcsIG1vZGVsIENQVSBvZmZsb2FkKTogNjMtNzEgcy9zdGVwLCAxNC4wIEdCIHBlYWsgLT4KIyAgIH4yNCBtaW4gcGVyIGNsaXAsIGFib3J0ZWQgYnkgdGhlIGd1YXJkLiBOb3c6IHByb21wdHMgZW5jb2RlZCBmaXJzdCAoVU1UNSBmcmVlZCksIHdob2xlIDEuM0IgbW9kZWwgcmVzaWRlbnQgb24KIyAgIHRoZSBHUFUgKG5vIG9mZmxvYWQpLCA0OSBmcmFtZXMgYnkgZGVmYXVsdCwgb3B0aW9uYWwgQ0ZHIGN1dC1vZmYgKGNmZ191bnRpbDogZnJhY3Rpb24gb2Ygc3RlcHMgdGhhdCB1c2UgQ0ZHOyB0aGUKIyAgIHJlc3QgcnVuIG9uZSBwYXNzIGluc3RlYWQgb2YgdHdvKSwgb3B0aW9uYWwgRmlyc3QtQmxvY2stQ2FjaGUgKGNhY2hlOiB0aHJlc2hvbGQsIGRpZmZ1c2VycyBidWlsdC1pbiwgbm8gbmV3CiMgICBtb2RlbCksIG9uZSBjbGlwIHBlciBUNCAoMiBpbiBwYXJhbGxlbCksIG1lbW9yeSBmcmVlZCBhZnRlciBhIGZhaWxlZCBjbGlwLiBwcm9iZSBpdGVtcyAocHJvYmU6IHRydWUpIHRpbWUgYSBmZXcKIyAgIHN0ZXBzIG9mIGEgY29uZmlndXJhdGlvbiB3aXRob3V0IGRlY29kaW5nLCBmb3IgYmVuY2htYXJrcy4KIwojIFByZXNldHMgKHBlci1pdGVtIHdpZHRoL2hlaWdodC9mcmFtZXMgb3ZlcnJpZGUgdGhlbSkuIFdhbiBnZW5lcmF0ZXMgMTYgZnBzOyB2cy1wcm9kdWNlIGNvbmZvcm1zIHRoZSBmcmFtZSByYXRlLgojICAgdDQgICAgICAgODMyeDQ4MCwgNDkgZnJhbWVzICgzLjEgcyBAMTYpLCAyMCBzdGVwcyAg4oCUIGRlZmF1bHQgb24gVDQKIyAgIHQ0LWxvbmcgIDgzMng0ODAsIDgxIGZyYW1lcyAoNS4xIHMgQDE2KSwgMjAgc3RlcHMgIOKAlCB+MS43eCB0aGUgdGltZSBvZiB0NAojICAgdDQtZmFzdCAgODMyeDQ4MCwgMzMgZnJhbWVzICgyLjEgcyBAMTYpLCAxNiBzdGVwcyAg4oCUIGRyYWZ0cyBhbmQgc2hvcnQgY3V0YXdheXMKIyAgIGhxICAgICAgIDEyODB4NzA0LCAxMjEgZnJhbWVzLCAzMCBzdGVwcyAgICAgICAgICAgIOKAlCBvbmx5IG9uIGJpZ2dlciBHUFVzIChMNC9BMTAwKTsgaG91cnMgcGVyIGNsaXAgb24gVDQKaW1wb3J0IG9zLCBzdWJwcm9jZXNzLCBzeXMKb3MuZW52aXJvbi5zZXRkZWZhdWx0KCJQWVRPUkNIX0NVREFfQUxMT0NfQ09ORiIsICJleHBhbmRhYmxlX3NlZ21lbnRzOlRydWUiKQpzdWJwcm9jZXNzLnJ1bihbc3lzLmV4ZWN1dGFibGUsICItbSIsICJwaXAiLCAiaW5zdGFsbCIsICItcSIsICJkaWZmdXNlcnM9PTAuNDMuMCIsICJhY2NlbGVyYXRlIiwgInRyYW5zZm9ybWVycyIsICJmdGZ5IiwKICAgICAgICAgICAgICAgICJpbWFnZWlvLWZmbXBlZyJdLCBjaGVjaz1UcnVlKQppbXBvcnQgdG9yY2gKZnJvbSBkaWZmdXNlcnMudXRpbHMgaW1wb3J0IGV4cG9ydF90b192aWRlbywgbG9hZF9pbWFnZQoKUFJFU0VUUyA9IHsidDQiOiAoODMyLCA0ODAsIDQ5LCAyMCksICJ0NC1sb25nIjogKDgzMiwgNDgwLCA4MSwgMjApLCAidDQtZmFzdCI6ICg4MzIsIDQ4MCwgMzMsIDE2KSwKICAgICAgICAgICAiaHEiOiAoMTI4MCwgNzA0LCAxMjEsIDMwKX0KUkVQT1MgPSB7IndhbjIyLTViIjogKCJXYW4tQUkvV2FuMi4yLVRJMlYtNUItRGlmZnVzZXJzIiwgImI4ZmZmNzMxNWM3NiIpLAogICAgICAgICAid2FuMjEtMS4zYiI6ICgiV2FuLUFJL1dhbjIuMS1UMlYtMS4zQi1EaWZmdXNlcnMiLCAiMGZhZDc4MGE1MzRiIil9Cm1vZGVsID0gUEFSQU1TLmdldCgibW9kZWwiKSBvciAoIndhbjIxLTEuM2IiIGlmIGlzX3Q0KCkgZWxzZSAid2FuMjItNWIiKQpwcmVzZXQgPSBQQVJBTVMuZ2V0KCJwcmVzZXQiLCAidDQiIGlmIGlzX3Q0KCkgZWxzZSAiaHEiKQpQVywgUEgsIFBGLCBQUyA9IFBSRVNFVFNbcHJlc2V0XQpORUcgPSAiYmx1cnJ5LCBsb3cgcXVhbGl0eSwgZGlzdG9ydGVkIGZhY2UsIGV4dHJhIGxpbWJzLCB3YXRlcm1hcmssIHRleHQsIHN1YnRpdGxlcywgc3RhdGljIGZyYW1lLCBmbGlja2VyIgppZiBtb2RlbCA9PSAid2FuMjEtMS4zYiIgYW5kIGFueShpdC5nZXQoImltYWdlIikgZm9yIGl0IGluIFBBUkFNU1siaXRlbXMiXSk6CiAgICBpZiBQQVJBTVMuZ2V0KCJhbGxvd19oZWF2eSIpOgogICAgICAgIG1vZGVsID0gIndhbjIyLTViIiAgICAgICAgICAgICAgICAgICAjIG9ubHkgdGhlIDVCIG1vZGVsIGNhbiBhbmltYXRlIGEga2V5ZnJhbWUKICAgICAgICByZXBvcnQobm90ZT0iaW1hZ2UtdG8tdmlkZW8gcmVxdWVzdGVkOiB1c2luZyB3YW4yMi01YiAoYWxsb3dfaGVhdnkpIikKICAgIGVsc2U6CiAgICAgICAgcmVwb3J0KG5vdGU9IndhbjIxLTEuM2IgaXMgdGV4dC1vbmx5OiBrZXlmcmFtZXMgaWdub3JlZCAoc2V0IGFsbG93X2hlYXZ5IGZvciB3YW4yMi01YiBpbWFnZS10by12aWRlbykiKQpJVEVNUyA9IFBBUkFNU1siaXRlbXMiXQppZiBtb2RlbCA9PSAid2FuMjEtMS4zYiIgYW5kIGxlbihJVEVNUykgPiAxOgogICAgSVRFTVMgPSBzaGFyZChJVEVNUykgICAgICAgICAgICAgICAgICMgb25lIHByb2Nlc3MgcGVyIFQ0OiB0d28gY2xpcHMgYXQgb25jZSAocGFyZW50IHdhaXRzLCB0aGVuIHN0b3BzIGhlcmUpCgoKZGVmIGhvc3RfcmFtX2diKCk6CiAgICB0cnk6CiAgICAgICAgcmV0dXJuIG9zLnN5c2NvbmYoIlNDX1BBR0VfU0laRSIpICogb3Muc3lzY29uZigiU0NfUEhZU19QQUdFUyIpIC8gMioqMzAKICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgT1NFcnJvcik6CiAgICAgICAgcmV0dXJuIDY0LjAKCgpMT1dfUkFNID0gYm9vbChQQVJBTVMuZ2V0KCJsb3dfcmFtIiwgaG9zdF9yYW1fZ2IoKSA8IDIwKSkKVEVYVF9GSVJTVCA9IExPV19SQU0gb3IgbW9kZWwgPT0gIndhbjIxLTEuM2IiICAgIyBlbmNvZGUgcHJvbXB0cywgZnJlZSB0aGUgMTEgR0IgZW5jb2RlciwgdGhlbiBsb2FkIHRoZSByZXN0CkVNQiA9IHt9CgoKZGVmIHByZV9lbmNvZGUoZHR5cGUpOgogICAgIiIiRW5jb2RlIGV2ZXJ5IHByb21wdCArIG5lZ2F0aXZlIHdpdGggVU1UNSBvbiB0aGUgR1BVLCB0aGVuIGZyZWUgaXQgKGxvdy1SQU0gaG9zdHMpLiIiIgogICAgZnJvbSBkaWZmdXNlcnMgaW1wb3J0IFdhblBpcGVsaW5lCiAgICBmcm9tIHRyYW5zZm9ybWVycyBpbXBvcnQgVU1UNUVuY29kZXJNb2RlbAogICAgcmVwbywgcmV2ID0gUkVQT1NbbW9kZWxdCiAgICB0ZV9kdHlwZSA9IHRvcmNoLmJmbG9hdDE2IGlmIHRvcmNoLmN1ZGEuaXNfYmYxNl9zdXBwb3J0ZWQoaW5jbHVkaW5nX2VtdWxhdGlvbj1UcnVlKSBlbHNlIGR0eXBlCiAgICB0ZSA9IFVNVDVFbmNvZGVyTW9kZWwuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHN1YmZvbGRlcj0idGV4dF9lbmNvZGVyIiwgcmV2aXNpb249cmV2LCB0b3JjaF9kdHlwZT10ZV9kdHlwZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZGV2aWNlX21hcD0iY3VkYSIsIGxvd19jcHVfbWVtX3VzYWdlPVRydWUpCiAgICBlbmMgPSBXYW5QaXBlbGluZS5mcm9tX3ByZXRyYWluZWQocmVwbywgcmV2aXNpb249cmV2LCB0ZXh0X2VuY29kZXI9dGUsIHRyYW5zZm9ybWVyPU5vbmUsIHZhZT1Ob25lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRvcmNoX2R0eXBlPXRlX2R0eXBlKQogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgZm9yIGl0IGluIElURU1TOgogICAgICAgICAgICBwZSwgbmUgPSBlbmMuZW5jb2RlX3Byb21wdChwcm9tcHQ9aXRbInByb21wdCJdLCBuZWdhdGl2ZV9wcm9tcHQ9aXQuZ2V0KCJuZWdhdGl2ZSIsIE5FRyksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGRvX2NsYXNzaWZpZXJfZnJlZV9ndWlkYW5jZT1UcnVlLCBudW1fdmlkZW9zX3Blcl9wcm9tcHQ9MSwgZGV2aWNlPSJjdWRhIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X3NlcXVlbmNlX2xlbmd0aD01MTIpICAgIyA9IFdhblBpcGVsaW5lLl9fY2FsbF9fIGRlZmF1bHQKICAgICAgICAgICAgaWYgbm90ICh0b3JjaC5pc2Zpbml0ZShwZSkuYWxsKCkgYW5kIHRvcmNoLmlzZmluaXRlKG5lKS5hbGwoKSk6CiAgICAgICAgICAgICAgICByYWlzZSBGbG9hdGluZ1BvaW50RXJyb3IoZiJ7aXRbJ2lkJ119OiBub24tZmluaXRlIHRleHQgZW1iZWRkaW5ncyIpCiAgICAgICAgICAgIEVNQltpdFsiaWQiXV0gPSAocGUudG8oImNwdSIpLCBuZS50bygiY3B1IikpCiAgICBkZWwgZW5jLCB0ZQogICAgaW1wb3J0IGdjOyBnYy5jb2xsZWN0KCk7IHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQoKCmRlZiBtYWtlX2kydih0MnYpOgogICAgIiIiSW1hZ2UtdG8tdmlkZW8gcGlwZWxpbmUgc2hhcmluZyB0MnYncyB3ZWlnaHRzLiBgY29tcG9uZW50c2AgZHJvcHMgY29uZmlnIGZsYWdzLCBhbmQgV2FuMi4yLVRJMlYtNUIgbmVlZHMKICAgIGV4cGFuZF90aW1lc3RlcHM9VHJ1ZSAoZmlyc3QtZnJhbWUgbGF0ZW50IGNvbmRpdGlvbmluZyk7IHdpdGhvdXQgaXQgZGlmZnVzZXJzIHRha2VzIHRoZSBXYW4yLjEgcGF0aCBhbmQgdGhlCiAgICA1QiB0cmFuc2Zvcm1lciBjcmFzaGVzIG9uIGEgMTAwLXZzLTQ4LWNoYW5uZWwgaW5wdXQgKHJlcHJvZHVjZWQgb24gQ1BVIHdpdGggdGlueSBtb2RlbHMsIGRpZmZ1c2VycyAwLjQzKS4iIiIKICAgIGlmIG1vZGVsICE9ICJ3YW4yMi01YiI6CiAgICAgICAgcmV0dXJuIE5vbmUKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBXYW5JbWFnZVRvVmlkZW9QaXBlbGluZQogICAgcmV0dXJuIFdhbkltYWdlVG9WaWRlb1BpcGVsaW5lKCoqdDJ2LmNvbXBvbmVudHMsIGV4cGFuZF90aW1lc3RlcHM9dDJ2LmNvbmZpZy5leHBhbmRfdGltZXN0ZXBzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGJvdW5kYXJ5X3JhdGlvPXQydi5jb25maWcuYm91bmRhcnlfcmF0aW8pCgoKZGVmIGJ1aWxkKGR0eXBlKToKICAgIGZyb20gZGlmZnVzZXJzIGltcG9ydCBXYW5QaXBlbGluZSwgV2FuSW1hZ2VUb1ZpZGVvUGlwZWxpbmUsIEF1dG9lbmNvZGVyS0xXYW4KICAgIHJlcG8sIHJldiA9IFJFUE9TW21vZGVsXQogICAgdmFlID0gQXV0b2VuY29kZXJLTFdhbi5mcm9tX3ByZXRyYWluZWQocmVwbywgc3ViZm9sZGVyPSJ2YWUiLCByZXZpc2lvbj1yZXYsIHRvcmNoX2R0eXBlPXRvcmNoLmZsb2F0MzIpCiAgICBpZiBURVhUX0ZJUlNUOgogICAgICAgIGlmIG5vdCBFTUI6CiAgICAgICAgICAgIHQgPSB0aW1lLnRpbWUoKTsgcHJlX2VuY29kZShkdHlwZSk7IHJlcG9ydChlbmNvZGVfcz1yb3VuZCh0aW1lLnRpbWUoKSAtIHQsIDEpKQogICAgICAgIGlmIG1vZGVsID09ICJ3YW4yMS0xLjNiIjogICAgICAgIyAyLjYgR0IgdHJhbnNmb3JtZXIgKyBWQUUgc3RheSBvbiB0aGUgR1BVOiBubyBDUFUgcm91bmQtdHJpcHMgcGVyIHN0ZXAKICAgICAgICAgICAgdDJ2ID0gV2FuUGlwZWxpbmUuZnJvbV9wcmV0cmFpbmVkKHJlcG8sIHJldmlzaW9uPXJldiwgdmFlPXZhZSwgdGV4dF9lbmNvZGVyPU5vbmUsIHRvcmNoX2R0eXBlPWR0eXBlKS50bygiY3VkYSIpCiAgICAgICAgICAgIHJlcG9ydChwbGFjZW1lbnQ9Indob2xlIG1vZGVsIG9uIG9uZSBUNCAocHJvbXB0cyBlbmNvZGVkIGZpcnN0LCB0ZXh0IGVuY29kZXIgZnJlZWQpIikKICAgICAgICBlbHNlOgogICAgICAgICAgICB0MnYgPSBsb2FkX3BpcGUoV2FuUGlwZWxpbmUsIHJlcG8sIHJldiwgZHR5cGUsIGZwOF9vaz1GYWxzZSwgcHJlZmVyPVBBUkFNUy5nZXQoInBsYWNlbWVudCIsICJvZmZsb2FkIiksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICB2YWU9dmFlLCB0ZXh0X2VuY29kZXI9Tm9uZSkKICAgIGVsc2U6CiAgICAgICAgIyBNZWFzdXJlZCBvbiAyeCBUNCAoYmFsYW5jZWQgKyBmcDgpOiBkZW5vaXNlIDcuOCBzL3N0ZXAsIGJ1dCB0aGUgVkFFIGxhbmRlZCBvZmYtR1BVIChkZWNvZGUgfjYyIG1pbikgYW5kCiAgICAgICAgIyBmcDE2IG91dHB1dCB3YXMgYmxhY2suIFNvOiBvbmUgVDQgd2l0aCBtb2RlbCBDUFUgb2ZmbG9hZCwgVkFFIG9uIHRoZSBHUFUsIG5vIGZwOC4KICAgICAgICB0MnYgPSBsb2FkX3BpcGUoV2FuUGlwZWxpbmUsIHJlcG8sIHJldiwgZHR5cGUsIGZwOF9vaz1GYWxzZSwgcHJlZmVyPVBBUkFNUy5nZXQoInBsYWNlbWVudCIsICJvZmZsb2FkIiksIHZhZT12YWUpCiAgICB2YWVfdGlsaW5nKHQydikKICAgIHJldHVybiB0MnYsIG1ha2VfaTJ2KHQydikKCgpDQUNIRSA9IHsidGhyIjogTm9uZX0KCgpkZWYgc2V0X2NhY2hlKHRyLCB0aHIpOgogICAgIiIiRmlyc3QtQmxvY2stQ2FjaGUgKGRpZmZ1c2VycyBidWlsdC1pbik6IHNraXAgdGhlIHJlc3Qgb2YgdGhlIHRyYW5zZm9ybWVyIHdoZW4gdGhlIGZpcnN0IGJsb2NrIGJhcmVseSBjaGFuZ2VzLiIiIgogICAgaWYgdGhyID09IENBQ0hFWyJ0aHIiXToKICAgICAgICBpZiB0aHI6CiAgICAgICAgICAgIHRyLl9yZXNldF9zdGF0ZWZ1bF9jYWNoZSgpCiAgICAgICAgcmV0dXJuCiAgICB0cnk6CiAgICAgICAgdHIuZGlzYWJsZV9jYWNoZSgpCiAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgIHBhc3MKICAgIGlmIHRocjoKICAgICAgICBmcm9tIGRpZmZ1c2Vycy5ob29rcyBpbXBvcnQgRmlyc3RCbG9ja0NhY2hlQ29uZmlnCiAgICAgICAgdHIuZW5hYmxlX2NhY2hlKEZpcnN0QmxvY2tDYWNoZUNvbmZpZyh0aHJlc2hvbGQ9ZmxvYXQodGhyKSkpCiAgICBDQUNIRVsidGhyIl0gPSB0aHIKCgpkZWYgc25hcCh2LCBtKToKICAgIHJldHVybiBtYXgobSwgaW50KHJvdW5kKHYgLyBtKSkgKiBtKQoKCkxBU1QgPSB7fQoKCmRlZiByZW5kZXIocGlwZXMsIGl0LCBkdHlwZSk6CiAgICB0MnYsIGkydiA9IHBpcGVzCiAgICB3LCBoID0gaXQuZ2V0KCJ3aWR0aCIsIFBXKSwgaXQuZ2V0KCJoZWlnaHQiLCBQSCkKICAgIGlmIGggPiB3IGFuZCAodywgaCkgPT0gKFBXLCBQSCk6CiAgICAgICAgdywgaCA9IGgsIHcgICAgICAgICAgICAgICAgICAgICAgICAgICMgdmVydGljYWwKICAgIGlmICJ3aWR0aCIgbm90IGluIGl0IGFuZCAiaGVpZ2h0IiBub3QgaW4gaXQgYW5kIGl0LmdldCgidmVydGljYWwiKToKICAgICAgICB3LCBoID0gUEgsIFBXCiAgICB3LCBoID0gc25hcCh3LCAzMiksIHNuYXAoaCwgMzIpCiAgICBmID0gaXQuZ2V0KCJmcmFtZXMiLCBQRik7IGYgPSAoZiAtIDEpIC8vIDQgKiA0ICsgMQogICAgaWYgVEVYVF9GSVJTVDogICAgICAgICAgICAgICAgICAgICAgICAgICAjIGVtYmVkZGluZ3MgbWFkZSBieSBwcmVfZW5jb2RlICh0ZXh0IGVuY29kZXIgYWxyZWFkeSBmcmVlZCkKICAgICAgICBwZSwgbmUgPSBFTUJbaXRbImlkIl1dCiAgICAgICAgdHh0ID0gZGljdChwcm9tcHRfZW1iZWRzPXBlLnRvKCJjdWRhIiwgZHR5cGUpLCBuZWdhdGl2ZV9wcm9tcHRfZW1iZWRzPW5lLnRvKCJjdWRhIiwgZHR5cGUpKQogICAgZWxzZToKICAgICAgICB0eHQgPSBkaWN0KHByb21wdD1pdFsicHJvbXB0Il0sIG5lZ2F0aXZlX3Byb21wdD1pdC5nZXQoIm5lZ2F0aXZlIiwgTkVHKSkKICAgIHN0ZXBzID0gUEFSQU1TLmdldCgicHJvYmVfc3RlcHMiLCA0KSBpZiBpdC5nZXQoInByb2JlIikgZWxzZSBpdC5nZXQoInN0ZXBzIiwgUEFSQU1TLmdldCgic3RlcHMiLCBQUykpCiAgICBjZmcgPSBpdC5nZXQoImNmZyIsIDUuMCkKICAgIGNmZ19mcmFjID0gaXQuZ2V0KCJjZmdfdW50aWwiLCBQQVJBTVMuZ2V0KCJjZmdfdW50aWwiLCAwLjUgaWYgaXNfdDQoKSBlbHNlIDEuMCkpICAgIyBtZWFzdXJlZCBmaW5lIG9uIFQ0OiAwLjUKICAgIGN1dCA9IGludChyb3VuZChzdGVwcyAqIGZsb2F0KGNmZ19mcmFjKSkpIGlmIGNmZyA+IDEgZWxzZSAwCiAgICBrdyA9IGRpY3QoKip0eHQsIGhlaWdodD1oLCB3aWR0aD13LCBudW1fZnJhbWVzPWYsIGd1aWRhbmNlX3NjYWxlPWNmZyBpZiBjdXQgZWxzZSAxLjAsCiAgICAgICAgICAgICAgbnVtX2luZmVyZW5jZV9zdGVwcz1zdGVwcywgZ2VuZXJhdG9yPXRvcmNoLkdlbmVyYXRvcigiY3B1IikubWFudWFsX3NlZWQoaXQuZ2V0KCJzZWVkIiwgMCkpKQogICAgaWYgaXQuZ2V0KCJwcm9iZSIpOgogICAgICAgIGt3WyJvdXRwdXRfdHlwZSJdID0gImxhdGVudCIKICAgIHdlaWdodHMgPSBbMiBpZiBpIDwgY3V0IGVsc2UgMSBmb3IgaSBpbiByYW5nZShzdGVwcyldCiAgICBnID0gc3RlcF9ndWFyZChzdGVwcywgaXQuZ2V0KCJidWRnZXRfcyIsIFBBUkFNUy5nZXQoIml0ZW1fYnVkZ2V0X3MiLCAxNTAwKSksIGl0WyJpZCJdLCB3ZWlnaHRzKQoKICAgIGRlZiBjYihwaXBlLCBpLCB0LCBrKToKICAgICAgICBrID0gZyhwaXBlLCBpLCB0LCBrKQogICAgICAgIGlmIGkgKyAxID49IGN1dDoKICAgICAgICAgICAgcGlwZS5fZ3VpZGFuY2Vfc2NhbGUgPSAxLjAgICAgICAgIyBDRkcgb2ZmIGZvciB0aGUgcmVtYWluaW5nIHN0ZXBzOiBvbmUgdHJhbnNmb3JtZXIgcGFzcyBpbnN0ZWFkIG9mIHR3bwogICAgICAgIHJldHVybiBrCiAgICBrd1siY2FsbGJhY2tfb25fc3RlcF9lbmQiXSA9IGNiOyBMQVNULnVwZGF0ZShndWFyZD1nLCBwYXNzZXM9c3VtKHdlaWdodHMpLCBjdXQ9Y3V0LCBzdGVwcz1zdGVwcykKICAgIHNldF9jYWNoZSh0MnYudHJhbnNmb3JtZXIsIGl0LmdldCgiY2FjaGUiLCBQQVJBTVMuZ2V0KCJjYWNoZSIpKSkKICAgIGlmIGl0LmdldCgiaW1hZ2UiKSBhbmQgaTJ2IGlzIG5vdCBOb25lIGFuZCBtb2RlbCA9PSAid2FuMjItNWIiOgogICAgICAgIGZyYW1lcyA9IGkydihpbWFnZT1sb2FkX2ltYWdlKHN0cihJTiAvIGl0WyJpbWFnZSJdKSkucmVzaXplKCh3LCBoKSksICoqa3cpLmZyYW1lc1swXQogICAgZWxzZToKICAgICAgICBmcmFtZXMgPSB0MnYoKiprdykuZnJhbWVzWzBdCiAgICByZXR1cm4gZnJhbWVzLCAodywgaCwgZikKCgpkZWYgdGltaW5nKHRvdGFsKToKICAgIHN0ID0gTEFTVFsiZ3VhcmQiXS5zdGF0cwogICAgcmV0dXJuIHsic3RlcHMiOiBMQVNUWyJzdGVwcyJdLCAiY2ZnX3N0ZXBzIjogTEFTVFsiY3V0Il0sICJjYWNoZSI6IENBQ0hFWyJ0aHIiXSwgImRlbm9pc2VfcyI6IHN0WyJkZW5vaXNlX3MiXSwKICAgICAgICAgICAgInNfcGVyX3N0ZXAiOiByb3VuZChzdFsiZGVub2lzZV9zIl0gLyBtYXgoMSwgc3RbInN0ZXBzIl0pLCAyKSwKICAgICAgICAgICAgInNfcGVyX3Bhc3MiOiByb3VuZChzdFsiZGVub2lzZV9zIl0gLyBtYXgoMSwgTEFTVFsicGFzc2VzIl0pLCAyKSwgImRlY29kZV9zIjogcm91bmQodG90YWwgLSBzdFsiZGVub2lzZV9zIl0sIDEpLAogICAgICAgICAgICAidnJhbV9nYiI6IHJvdW5kKHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSAvIDIqKjMwLCAyKX0KCgpkZWYgZnJlZSgpOgogICAgaW1wb3J0IGdjCiAgICBnYy5jb2xsZWN0KCk7IHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKTsgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCgoKdCA9IHRpbWUudGltZSgpCmR0eXBlID0gcGlja19kdHlwZShmcDE2X3NhZmU9VHJ1ZSkKcGlwZXMgPSBidWlsZChkdHlwZSkKcmVwb3J0KGxvYWRfcz1yb3VuZCh0aW1lLnRpbWUoKSAtIHQsIDEpLCBkdHlwZT1zdHIoZHR5cGUpLCBtb2RlbD1tb2RlbCwgcHJlc2V0PXByZXNldCwgbG93X3JhbT1MT1dfUkFNLAogICAgICAgaG9zdF9yYW1fZ2I9cm91bmQoaG9zdF9yYW1fZ2IoKSwgMSkpCmRvbmUsIHJlZG8sIGZhaWxlZCwgcHJvYmVzID0gW10sIFtdLCBbXSwgW10KdHJ5OgogICAgZnJvbSB0b3JjaC5ubi5hdHRlbnRpb24gaW1wb3J0IFNEUEJhY2tlbmQgICMgd2hpY2ggYXR0ZW50aW9uIGtlcm5lbHMgdGhpcyB0b3JjaCBidWlsZCBvZmZlcnMgb24gVDQKICAgIHJlcG9ydChzZHBhPXsiZmxhc2giOiB0b3JjaC5iYWNrZW5kcy5jdWRhLmZsYXNoX3NkcF9lbmFibGVkKCksICJtZW1fZWZmaWNpZW50IjogdG9yY2guYmFja2VuZHMuY3VkYS5tZW1fZWZmaWNpZW50X3NkcF9lbmFibGVkKCksCiAgICAgICAgICAgICAgICAgInRvcmNoIjogdG9yY2guX192ZXJzaW9uX199KQpleGNlcHQgRXhjZXB0aW9uOgogICAgcGFzcwpmb3IgaXQgaW4gSVRFTVM6CiAgICBzID0gdGltZS50aW1lKCk7IGZyZWUoKQogICAgdHJ5OgogICAgICAgIGZyYW1lcywgZGltcyA9IHJlbmRlcihwaXBlcywgaXQsIGR0eXBlKQogICAgZXhjZXB0IEZsb2F0aW5nUG9pbnRFcnJvciBhcyBlOgogICAgICAgIHJlZG8uYXBwZW5kKGl0KTsgcmVwb3J0KGNsaXBzPWRvbmUsIGJyb2tlbl9mcDE2PVtyWyJpZCJdIGZvciByIGluIHJlZG9dLCBuYW49c3RyKGUpKTsgZnJlZSgpOyBjb250aW51ZQogICAgZXhjZXB0IFRpbWVvdXRFcnJvciBhcyBlOiAgICAgICAgICAgICAgICAjIHRvbyBzbG93IGZvciB0aGlzIGhhcmR3YXJlOiBnaXZlIHVwIG9uIHRoaXMgY2xpcCwga2VlcCBxdW90YQogICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5Ijogc3RyKGUpfSk7IHJlcG9ydChjbGlwcz1kb25lLCBmYWlsZWQ9ZmFpbGVkKTsgZnJlZSgpOyBjb250aW51ZQogICAgZXhjZXB0IHRvcmNoLk91dE9mTWVtb3J5RXJyb3IgYXMgZToKICAgICAgICBmYWlsZWQuYXBwZW5kKHsiaWQiOiBpdFsiaWQiXSwgIndoeSI6ICJvdXQgb2YgR1BVIG1lbW9yeTogIiArIHN0cihlKVs6MTYwXX0pOyByZXBvcnQoY2xpcHM9ZG9uZSwgZmFpbGVkPWZhaWxlZCkKICAgICAgICBmcmVlKCk7IGNvbnRpbnVlCiAgICBpZiBpdC5nZXQoInByb2JlIik6CiAgICAgICAgcHJvYmVzLmFwcGVuZCh7ImlkIjogaXRbImlkIl0sICJzaXplIjogZiJ7ZGltc1swXX14e2RpbXNbMV19IiwgImZyYW1lcyI6IGRpbXNbMl0sICoqdGltaW5nKHRpbWUudGltZSgpIC0gcyl9KQogICAgICAgIHJlcG9ydChwcm9iZXM9cHJvYmVzKTsgY29udGludWUKICAgIGlmIGxvb2tzX2Jyb2tlbihmcmFtZXNbbGVuKGZyYW1lcykgLy8gMl0pOiAgICAjIGtlZXAgdGhlIGV2aWRlbmNlOiBzYXZlZCBhcyAuc3VzcGVjdC5tcDQgZm9yIGluc3BlY3Rpb24KICAgICAgICBleHBvcnRfdG9fdmlkZW8oZnJhbWVzLCBzdHIoT1VUIC8gZiJ7aXRbJ2lkJ119LnN1c3BlY3QubXA0IiksIGZwcz1QQVJBTVMuZ2V0KCJmcHMiLCAxNikpCiAgICAgICAgcmVkby5hcHBlbmQoaXQpOyByZXBvcnQoY2xpcHM9ZG9uZSwgYnJva2VuX2ZwMTY9W3JbImlkIl0gZm9yIHIgaW4gcmVkb10pOyBjb250aW51ZQogICAgZXhwb3J0X3RvX3ZpZGVvKGZyYW1lcywgc3RyKE9VVCAvIGYie2l0WydpZCddfS5tcDQiKSwgZnBzPVBBUkFNUy5nZXQoImZwcyIsIDE2KSkKICAgIGRvbmUuYXBwZW5kKHsiaWQiOiBpdFsiaWQiXSwgInMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHMsIDEpLCAic2l6ZSI6IGYie2RpbXNbMF19eHtkaW1zWzFdfSIsICJmcmFtZXMiOiBkaW1zWzJdLAogICAgICAgICAgICAgICAgICoqdGltaW5nKHRpbWUudGltZSgpIC0gcyl9KQogICAgcmVwb3J0KGNsaXBzPWRvbmUpCmlmIHJlZG8gYW5kIGlzX3Q0KCk6ICAgICAgICAgICAgICAgICAgICAgICAgICMgbWVhc3VyZWQ6IGJmMTYgYXR0ZW50aW9uIG9uIFQ0IGhhcyBubyBmdXNlZCBrZXJuZWwgLT4gMTggR0IgbWF0cml4LCBPT00KICAgIGZvciBpdCBpbiByZWRvOgogICAgICAgIGZhaWxlZC5hcHBlbmQoeyJpZCI6IGl0WyJpZCJdLCAid2h5IjogImZwMTYgb3V0cHV0IGxvb2tlZCBicm9rZW47IGJmMTYgcmV0cnkgaW1wb3NzaWJsZSBvbiBUNCAoc2VlIC5zdXNwZWN0Lm1wNCkifSkKICAgIHJlcG9ydChmYWlsZWQ9ZmFpbGVkKTsgcmVkbyA9IFtdCmlmIHJlZG8gYW5kIGR0eXBlID09IHRvcmNoLmZsb2F0MTY6ICAgICAgICAgICMgZnAxNiBvdmVyZmxvd2VkIG9uIHRoZXNlIGNsaXBzOiByZWJ1aWxkIG9uY2UgaW4gYmYxNiAoc2xvd2VyLCBleGFjdCkKICAgIGZvciBwIGluIHBpcGVzOgogICAgICAgIGlmIHAgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRyeTogcC5yZW1vdmVfYWxsX2hvb2tzKCkKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbjogcGFzcwogICAgZGVsIHBpcGVzLCBwCiAgICBpbXBvcnQgZ2M7IGdjLmNvbGxlY3QoKTsgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICB0ID0gdGltZS50aW1lKCk7IHBpcGVzID0gYnVpbGQodG9yY2guYmZsb2F0MTYpOyByZXBvcnQoYmYxNl9sb2FkX3M9cm91bmQodGltZS50aW1lKCkgLSB0LCAxKSkKICAgIGZvciBpdCBpbiByZWRvOgogICAgICAgIHMgPSB0aW1lLnRpbWUoKQogICAgICAgIHRyeToKICAgICAgICAgICAgZnJhbWVzLCBkaW1zID0gcmVuZGVyKHBpcGVzLCBpdCwgdG9yY2guYmZsb2F0MTYpCiAgICAgICAgZXhjZXB0IChGbG9hdGluZ1BvaW50RXJyb3IsIFRpbWVvdXRFcnJvciwgdG9yY2guT3V0T2ZNZW1vcnlFcnJvcikgYXMgZToKICAgICAgICAgICAgZmFpbGVkLmFwcGVuZCh7ImlkIjogaXRbImlkIl0sICJ3aHkiOiAiYmYxNjogIiArIHN0cihlKVs6MjAwXX0pOyByZXBvcnQoZmFpbGVkPWZhaWxlZCk7IGZyZWUoKTsgY29udGludWUKICAgICAgICBleHBvcnRfdG9fdmlkZW8oZnJhbWVzLCBzdHIoT1VUIC8gZiJ7aXRbJ2lkJ119Lm1wNCIpLCBmcHM9UEFSQU1TLmdldCgiZnBzIiwgMTYpKQogICAgICAgIGRvbmUuYXBwZW5kKHsiaWQiOiBpdFsiaWQiXSwgInMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHMsIDEpLCAiZHR5cGUiOiAiYmYxNi1yZXRyeSIsICoqdGltaW5nKHRpbWUudGltZSgpIC0gcyl9KTsgcmVwb3J0KGNsaXBzPWRvbmUpCmlmIG5vdCBkb25lIGFuZCBub3QgcHJvYmVzOgogICAgcmVwb3J0KGVycm9yPSJubyBjbGlwIHdhcyBwcm9kdWNlZDogIiArICI7ICIuam9pbihmIntmWydpZCddfToge2ZbJ3doeSddfSIgZm9yIGYgaW4gZmFpbGVkKSkK\'\nexec(base64.b64decode(_COMMON_B64).decode())\n\n\ndef _env_check():  # did Kaggle grant what the job asked for? (GPU + internet need a phone-verified account)\n    import shutil as _sh, socket as _so, subprocess as _sp\n    gpu = bool(_sh.which("nvidia-smi")) and _sp.run(["nvidia-smi", "-L"], capture_output=True).returncode == 0\n    try:\n        _so.getaddrinfo("huggingface.co", 443); net = True\n    except OSError:\n        net = False\n    report(gpu_granted=gpu, internet_granted=net)\n    if PARAMS.get("_need_gpu") and not gpu:\n        report(error=("Kaggle gave this session no GPU: this accelerator is busy/unavailable or another GPU session "\n                      "is running (free accounts get one at a time); retry later.") if net else\n                     ("Kaggle gave this session no GPU and no internet: the account needs phone verification "\n                      "(kaggle.com > Settings > Phone verification)."), gpu_unavailable_reason="busy" if net else "unverified")\n        raise SystemExit(3)\n    if PARAMS.get("_need_internet") and not net:\n        report(error="Kaggle gave this session no internet (phone verification needed, or internet off).")\n        raise SystemExit(3)\n\n\n_env_check()\ntry:\n    import torch as _t\n    if _t.cuda.is_available():\n        report(gpu=gpu_info())\nexcept Exception:\n    pass\ntry:\n    exec(compile(base64.b64decode(_JOB_B64).decode(), "job.py", "exec"))\nexcept _Done:\n    pass\nif json.loads((OUT / "report.json").read_text()).get("error"):\n    raise SystemExit(4)\n'
open('/content/vs_job.py', 'w').write(JOB)
import subprocess, sys
rc = subprocess.run([sys.executable, '/content/vs_job.py']).returncode
print('job exit code', rc, '(0 = ok; see report.json in the zip either way)')


In [ ]:
# 4. pack the results and download them
import json, pathlib, shutil
out = pathlib.Path('/kaggle/working/out')
rep = out / 'report.json'
if rep.exists():
    d = json.loads(rep.read_text()); d['provider'] = 'colab'; rep.write_text(json.dumps(d, indent=2))
    print(json.dumps(d, indent=1)[:3000])
zp = shutil.make_archive('/content/vs-result-wan21-1-3b', 'zip', out)
from google.colab import files
files.download(zp)


In [ ]:
# 5. release the GPU (run after the download has started)
from google.colab import runtime
runtime.unassign()
